In [ ]:
# -*- coding: utf-8 -*-
"""
AUTOENCODER + COMPENSAÇÃO PARAMÉTRICA TÉRMICA — TREINO PESADO E CONSERVADOR

Este código foi feito para resolver o problema:
a referência saudável estava influenciando demais o FORMATO da curva compensada,
criando pontos extras e fazendo a curva parecer uma cópia da referência.

Aqui o autoencoder NÃO gera a curva compensada.

Fluxo correto:

1) Treina um autoencoder APENAS com curvas saudáveis.
   Ele serve só para comprimir a curva e extrair um estado térmico latente.

2) Para curvas saudáveis, calcula uma compensação térmica PARAMÉTRICA:

       y_comp(f) = a * x(f + tau) + b + c*z(f)

   onde:
       tau = deslocamento horizontal em frequência
       a   = ganho vertical
       b   = offset vertical
       c   = inclinação linear
       z   = frequência normalizada de -1 a 1

3) Treina um regressor simples:

       latente do AE  --->  [tau, a, b, c]

4) Para qualquer curva, inclusive com dano:

       curva original  --->  encoder do AE  --->  parâmetros térmicos
       curva compensada = transformação paramétrica da curva original

Ponto principal:
- A referência saudável NÃO entra somada ponto a ponto na curva final.
- O dano é preservado porque a transformação tem só 4 graus de liberdade.
- O método não consegue "desenhar" picos novos da referência.
- Esta versão treina mais, mas aplica uma compensação mais conservadora para preservar melhor a assinatura do dano.
"""

# ============================================================
# 1) IMPORTS
# ============================================================

import os
import re
import time
import copy
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler as SkStandardScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore", category=UserWarning)


# ============================================================
# 2) PARÂMETROS
# ============================================================

ARQ_BASE = "base-completo--.pkl"

REF_TEMP = 30

FREQ_MIN_KHZ = 40
FREQ_MAX_KHZ = 50

OUTPUT_DIR = (
    f"AE_PARAMETRICO_TERMICO_"
    f"{REF_TEMP}C_{FREQ_MIN_KHZ}-{FREQ_MAX_KHZ}kHz"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------- AUTOENCODER ----------------

EPOCHS = 900
BATCH_SIZE = 4
LR = 3e-4
PATIENCE = 150

# Gargalo pequeno para o AE não copiar detalhes locais.
LATENT_DIM = 8

INPUT_NOISE_STD = 0.02
DROPOUT = 0.15

LAMBDA_RECON = 1.00
LAMBDA_DERIV = 0.60
LAMBDA_CURVATURE = 0.35

NUM_WORKERS = 0

# ---------------- COMPENSAÇÃO PARAMÉTRICA ----------------

# Deslocamento máximo permitido em relação à largura da banda.
# Quanto menor, menos chance de deformar dano.
SHIFT_MAX_FRAC = 0.015

SHIFT_NSTEPS = 181

# Limites físicos/seguros dos parâmetros.
GAIN_MIN = 0.90
GAIN_MAX = 1.10

# Limites de offset e inclinação em fração da amplitude da referência.
OFFSET_FRAC_LIMIT = 0.22
TILT_FRAC_LIMIT = 0.14

# Quanto da compensação prevista aplicar.
# Se ainda deformar demais, teste 0.70 ou 0.80.
ALPHA_COMP = 0.70

# Regularização do Ridge que prediz parâmetros.
RIDGE_ALPHA = 12.0

# Interpretação rápida dos ajustes desta versão:
# - EPOCHS maior: treina melhor o autoencoder.
# - LATENT_DIM um pouco maior: representa melhor variações térmicas reais.
# - SHIFT/GAIN/OFFSET/TILT mais limitados: evita apagar assinatura de dano.
# - ALPHA_COMP menor: aplica só parte da correção térmica prevista.
# - RIDGE_ALPHA maior: deixa a predição dos parâmetros mais suave/conservadora.


# Modo:
# True  = usa temperatura medida para interpolar parâmetros saudáveis.
# False = usa latente do AE para predizer parâmetros.
#
# Para artigo/pesquisa com AE, deixe False.
# Para testar o limite físico do método, use True.
USE_KNOWN_TEMPERATURE_FOR_PARAMS = False

# ---------------- PARK OPCIONAL ----------------

USAR_PARK = True
PARK_MAX_SHIFT_FRAC = 0.10
PARK_NSTEPS = 101
PARK_SMOOTH_WIN = 1

# ---------------- FIGURAS ----------------

TEMP_ESCOLHIDA = 55
DANOS_PLOTAR = [0, 1, 2]
OCORRENCIA_CURVA = 0

TEMPERATURAS_BARRAS = None
N_TEMPS_BARRAS = 8

HIST_BINS = 18

np.random.seed(42)
torch.manual_seed(42)


# ============================================================
# 3) FUNÇÕES GERAIS
# ============================================================

def extract_freq_hz(col):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None


def get_freq_columns(df, fmin_khz, fmax_khz):
    cols = []
    freqs = []

    for c in df.columns:
        f = extract_freq_hz(c)

        if f is not None:
            f_khz = f / 1e3

            if fmin_khz <= f_khz <= fmax_khz:
                cols.append(c)
                freqs.append(f)

    order = np.argsort(freqs)

    fcols = [cols[i] for i in order]
    fhz = np.array(freqs, dtype=float)[order]

    return fcols, fhz


def aplicar_estilo_artigo():
    plt.rcParams.update({
        "font.family": "Times New Roman",
        "font.size": 18,
        "axes.labelsize": 20,
        "axes.titlesize": 20,
        "xtick.labelsize": 17,
        "ytick.labelsize": 17,
        "legend.fontsize": 14,
        "figure.dpi": 300,
        "savefig.dpi": 300,
        "pdf.fonttype": 42,
        "ps.fonttype": 42
    })


def salvar_figura(fig, nome_base, output_dir=OUTPUT_DIR, dpi=600):
    os.makedirs(output_dir, exist_ok=True)

    png_path = os.path.join(output_dir, f"{nome_base}.png")
    pdf_path = os.path.join(output_dir, f"{nome_base}.pdf")

    fig.savefig(png_path, dpi=dpi, bbox_inches="tight", facecolor="white")
    fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")

    print(f"Figura salva em PNG: {png_path}")
    print(f"Figura salva em PDF: {pdf_path}")

    return png_path, pdf_path


def formatar_temp(T):
    if float(T).is_integer():
        return f"{int(T)}"
    return f"{T:.1f}"


def moving_average(arr, win):
    arr = np.asarray(arr, dtype=float)

    if win <= 1:
        return arr.copy()

    if win % 2 == 0:
        win += 1

    pad = win // 2
    arr_pad = np.pad(arr, (pad, pad), mode="edge")
    kernel = np.ones(win) / win

    return np.convolve(arr_pad, kernel, mode="valid")


# ============================================================
# 4) REFERÊNCIAS SAUDÁVEIS
# ============================================================

def get_healthy_references_by_temperature(df, fcols, ref_temp):
    if "falha" not in df.columns:
        raise ValueError("O DataFrame precisa ter a coluna 'falha'.")

    if "temperatura_c" not in df.columns:
        raise ValueError("O DataFrame precisa ter a coluna 'temperatura_c'.")

    df_h = df[df["falha"] == 0].copy()

    if len(df_h) == 0:
        raise ValueError("Não há curvas saudáveis, isto é, falha = 0.")

    healthy_by_temp = {}

    temps_h = np.array(sorted(df_h["temperatura_c"].unique()), dtype=float)

    for T in temps_h:
        pool = df_h.loc[
            np.isclose(df_h["temperatura_c"], T),
            fcols
        ].to_numpy(float)

        healthy_by_temp[float(T)] = np.median(pool, axis=0)

    ref_temp_used = float(temps_h[np.argmin(np.abs(temps_h - ref_temp))])

    if not np.isclose(ref_temp_used, ref_temp):
        print(
            f"AVISO: não existe saudável exatamente em {ref_temp} °C. "
            f"Usando {ref_temp_used} °C como referência saudável."
        )

    y_ref_healthy = healthy_by_temp[ref_temp_used]

    return healthy_by_temp, temps_h, y_ref_healthy, ref_temp_used


def get_nearest_healthy_curve(healthy_by_temp, healthy_temps, temperatura):
    healthy_temps = np.asarray(healthy_temps, dtype=float)

    temp_used = float(
        healthy_temps[np.argmin(np.abs(healthy_temps - temperatura))]
    )

    return healthy_by_temp[temp_used], temp_used


# ============================================================
# 5) MÉTRICAS
# ============================================================

def rmsd(y, ref):
    y = np.asarray(y, dtype=float)
    ref = np.asarray(ref, dtype=float)

    return float(np.sqrt(np.mean((y - ref) ** 2)))


def ccdm(y, ref):
    y = np.asarray(y, dtype=float)
    ref = np.asarray(ref, dtype=float)

    y0 = y - np.mean(y)
    r0 = ref - np.mean(ref)

    num = float(np.sum(y0 * r0))
    den = float(np.sqrt(np.sum(y0 ** 2) * np.sum(r0 ** 2)) + 1e-18)

    corr = num / den

    return float(1 - corr)


def derivative_loss(y_pred, y_true):
    dy_pred = y_pred[:, 1:] - y_pred[:, :-1]
    dy_true = y_true[:, 1:] - y_true[:, :-1]

    return F.smooth_l1_loss(dy_pred, dy_true)


def curvature_loss(y_pred, y_true):
    d2_pred = y_pred[:, 2:] - 2 * y_pred[:, 1:-1] + y_pred[:, :-2]
    d2_true = y_true[:, 2:] - 2 * y_true[:, 1:-1] + y_true[:, :-2]

    return F.smooth_l1_loss(d2_pred, d2_true)


def calcular_metricas_com_preservacao(
    df_curvas,
    df_original,
    fcols,
    y_ref_healthy,
    healthy_by_temp,
    healthy_temps,
    metodo
):
    X_comp = df_curvas[fcols].to_numpy(float)
    X_orig = df_original[fcols].to_numpy(float)

    df_out = df_curvas.copy()

    rmsd_list = []
    ccdm_list = []

    damage_res_rmsd = []
    damage_res_ccdm = []

    alteracao_rmsd = []

    for i, (_, row) in enumerate(df_original.iterrows()):
        T = float(row["temperatura_c"])

        h_T, _ = get_nearest_healthy_curve(
            healthy_by_temp=healthy_by_temp,
            healthy_temps=healthy_temps,
            temperatura=T
        )

        y_comp = X_comp[i]
        y_orig = X_orig[i]

        rmsd_list.append(rmsd(y_comp, y_ref_healthy))
        ccdm_list.append(ccdm(y_comp, y_ref_healthy))

        assinatura_esperada = y_orig - h_T
        assinatura_saida = y_comp - y_ref_healthy

        damage_res_rmsd.append(rmsd(assinatura_saida, assinatura_esperada))
        damage_res_ccdm.append(ccdm(assinatura_saida, assinatura_esperada))

        alteracao_rmsd.append(rmsd(y_comp, y_orig))

    df_out["RMSD"] = rmsd_list
    df_out["CCDM"] = ccdm_list

    df_out["DamageResidual_RMSD"] = damage_res_rmsd
    df_out["DamageResidual_CCDM"] = damage_res_ccdm

    df_out["Alteracao_RMSD"] = alteracao_rmsd
    df_out["Metodo"] = metodo

    return df_out


# ============================================================
# 6) TRANSFORMAÇÃO PARAMÉTRICA
# ============================================================

def shift_interp(x, fHz, tau):
    """
    Retorna x deslocado no eixo de frequência.
    tau positivo/negativo desloca horizontalmente a curva.

    Importante:
    isso NÃO cria forma nova da referência.
    Apenas desloca a própria curva original.
    """

    f_shift = fHz + tau

    return np.interp(
        fHz,
        f_shift,
        x,
        left=x[0],
        right=x[-1]
    )


def aplicar_transformacao_parametrica(x, fHz, params):
    """
    y = a * x(f + tau) + b + c*z
    """

    tau, a, b, c = params

    z = np.linspace(-1.0, 1.0, len(x))

    x_shift = shift_interp(x, fHz, tau)

    y = a * x_shift + b + c * z

    return y


def limitar_parametros(params, ref_amp):
    tau, a, b, c = params

    a = float(np.clip(a, GAIN_MIN, GAIN_MAX))

    b_lim = OFFSET_FRAC_LIMIT * ref_amp
    c_lim = TILT_FRAC_LIMIT * ref_amp

    b = float(np.clip(b, -b_lim, b_lim))
    c = float(np.clip(c, -c_lim, c_lim))

    return np.array([tau, a, b, c], dtype=float)


def ajustar_params_healthy_para_ref(x_h, y_ref, fHz):
    """
    Calcula os parâmetros que levam uma curva saudável x_h até y_ref.

    Para cada tau testado, resolve por mínimos quadrados:

        y_ref ~= a*x_shift + b + c*z

    Depois escolhe o tau com menor erro.

    Isso é feito APENAS em curvas saudáveis.
    """

    df_band = fHz[-1] - fHz[0]
    tau_max = SHIFT_MAX_FRAC * df_band

    taus = np.linspace(-tau_max, tau_max, SHIFT_NSTEPS)

    z = np.linspace(-1.0, 1.0, len(x_h))

    best_err = np.inf
    best_params = np.array([0.0, 1.0, 0.0, 0.0], dtype=float)

    ref_amp = np.ptp(y_ref) + 1e-12

    for tau in taus:
        x_shift = shift_interp(x_h, fHz, tau)

        A = np.column_stack([
            x_shift,
            np.ones_like(x_shift),
            z
        ])

        coef, *_ = np.linalg.lstsq(A, y_ref, rcond=None)

        a, b, c = coef

        params = np.array([tau, a, b, c], dtype=float)
        params = limitar_parametros(params, ref_amp)

        y_try = aplicar_transformacao_parametrica(x_h, fHz, params)

        err = np.mean((y_try - y_ref) ** 2)

        if err < best_err:
            best_err = err
            best_params = params

    return best_params, best_err


def construir_dataset_parametros_saudaveis(df, fcols, fhz, y_ref_healthy):
    """
    Para cada curva saudável, calcula os parâmetros térmicos necessários
    para levar aquela curva para a referência.

    A curva danificada nunca entra aqui.
    """

    df_h = df[df["falha"] == 0].copy()

    Xh = df_h[fcols].to_numpy(float)
    Th = df_h["temperatura_c"].to_numpy(float)

    params_list = []
    errs = []
    idxs = []

    print("\n====================================================")
    print("AJUSTANDO PARÂMETROS TÉRMICOS EM CURVAS SAUDÁVEIS")
    print("====================================================")

    for k, idx in enumerate(df_h.index):
        x_h = df.loc[idx, fcols].to_numpy(float)

        params, err = ajustar_params_healthy_para_ref(
            x_h=x_h,
            y_ref=y_ref_healthy,
            fHz=fhz
        )

        params_list.append(params)
        errs.append(err)
        idxs.append(idx)

        if (k + 1) % 20 == 0 or (k + 1) == len(df_h):
            print(f"Parâmetros ajustados: {k+1}/{len(df_h)}")

    params_arr = np.vstack(params_list)
    errs = np.asarray(errs, dtype=float)

    out = pd.DataFrame({
        "index_original": idxs,
        "temperatura_c": Th,
        "tau": params_arr[:, 0],
        "gain": params_arr[:, 1],
        "offset": params_arr[:, 2],
        "tilt": params_arr[:, 3],
        "fit_mse": errs
    })

    return out


# ============================================================
# 7) AUTOENCODER SAUDÁVEL
# ============================================================

class HealthyAutoencoder(nn.Module):
    def __init__(self, n_points, latent_dim=6, dropout=0.20):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(n_points, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.GELU(),

            nn.Linear(64, latent_dim)
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.LayerNorm(64),
            nn.GELU(),

            nn.Linear(64, 128),
            nn.LayerNorm(128),
            nn.GELU(),

            nn.Linear(128, 256),
            nn.LayerNorm(256),
            nn.GELU(),

            nn.Linear(256, n_points)
        )

    def forward(self, x):
        z = self.encoder(x)
        y = self.decoder(z)

        return y, z


def treinar_autoencoder_saudavel(df, fcols):
    """
    Treina AE apenas com curvas saudáveis.
    A saída reconstruída NÃO será usada para compensar curva.
    Só usaremos o encoder.
    """

    print("\n====================================================")
    print("TREINANDO AUTOENCODER SAUDÁVEL")
    print("====================================================")
    print("Atenção: a saída do decoder NÃO gera curva compensada.")
    print("O AE será usado apenas como extrator latente térmico.")

    df_h = df[df["falha"] == 0].copy()

    Xh = df_h[fcols].to_numpy(float)

    scaler = StandardScaler()
    Xh_s = scaler.fit_transform(Xh)

    indices = np.arange(len(Xh_s))

    if len(indices) >= 10:
        idx_train, idx_val = train_test_split(
            indices,
            test_size=0.20,
            random_state=42
        )
    else:
        idx_train = indices
        idx_val = indices

    X_tensor = torch.tensor(Xh_s, dtype=torch.float32)

    train_loader = DataLoader(
        TensorDataset(X_tensor[idx_train]),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS
    )

    val_loader = DataLoader(
        TensorDataset(X_tensor[idx_val]),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS
    )

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Dispositivo usado: {device}")

    model = HealthyAutoencoder(
        n_points=Xh_s.shape[1],
        latent_dim=LATENT_DIM,
        dropout=DROPOUT
    ).to(device)

    opt = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=1e-4
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt,
        mode="min",
        factor=0.5,
        patience=35
    )

    huber = nn.SmoothL1Loss()

    best_val = np.inf
    best_state = copy.deepcopy(model.state_dict())
    epochs_sem_melhora = 0

    history = {
        "epoch": [],
        "train_loss": [],
        "val_loss": [],
        "val_recon": [],
        "lr": []
    }

    for ep in range(1, EPOCHS + 1):
        model.train()

        train_losses = []

        for (xb,) in train_loader:
            xb = xb.to(device)

            if INPUT_NOISE_STD > 0:
                xb_in = xb + INPUT_NOISE_STD * torch.randn_like(xb)
            else:
                xb_in = xb

            opt.zero_grad()

            pred, z = model(xb_in)

            loss_recon = huber(pred, xb)
            loss_deriv = derivative_loss(pred, xb)
            loss_curv = curvature_loss(pred, xb)

            loss = (
                LAMBDA_RECON * loss_recon
                + LAMBDA_DERIV * loss_deriv
                + LAMBDA_CURVATURE * loss_curv
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=5.0
            )

            opt.step()

            train_losses.append(loss.item())

        model.eval()

        val_losses = []
        val_recons = []

        with torch.no_grad():
            for (xb,) in val_loader:
                xb = xb.to(device)

                pred, z = model(xb)

                loss_recon = huber(pred, xb)
                loss_deriv = derivative_loss(pred, xb)
                loss_curv = curvature_loss(pred, xb)

                loss = (
                    LAMBDA_RECON * loss_recon
                    + LAMBDA_DERIV * loss_deriv
                    + LAMBDA_CURVATURE * loss_curv
                )

                val_losses.append(loss.item())
                val_recons.append(loss_recon.item())

        train_mean = float(np.mean(train_losses))
        val_mean = float(np.mean(val_losses))
        val_recon = float(np.mean(val_recons))

        scheduler.step(val_mean)

        current_lr = opt.param_groups[0]["lr"]

        history["epoch"].append(ep)
        history["train_loss"].append(train_mean)
        history["val_loss"].append(val_mean)
        history["val_recon"].append(val_recon)
        history["lr"].append(current_lr)

        if val_mean < best_val - 1e-7:
            best_val = val_mean
            best_state = copy.deepcopy(model.state_dict())
            epochs_sem_melhora = 0
        else:
            epochs_sem_melhora += 1

        if ep == 1 or ep % 25 == 0:
            print(
                f"Epoch {ep:4d}/{EPOCHS} | "
                f"train={train_mean:.6f} | "
                f"val={val_mean:.6f} | "
                f"recon={val_recon:.6f} | "
                f"lr={current_lr:.2e}"
            )

        if epochs_sem_melhora >= PATIENCE:
            print(
                f"Early stopping na epoch {ep}. "
                f"Melhor val_loss = {best_val:.6f}"
            )
            break

    model.load_state_dict(best_state)

    return {
        "model": model,
        "scaler": scaler,
        "device": device,
        "history": pd.DataFrame(history),
        "healthy_indices": df_h.index.to_numpy()
    }


def codificar_curvas_ae(df, fcols, ae_extra):
    model = ae_extra["model"]
    scaler = ae_extra["scaler"]
    device = ae_extra["device"]

    X = df[fcols].to_numpy(float)
    Xs = scaler.transform(X)

    X_tensor = torch.tensor(Xs, dtype=torch.float32)

    model.eval()

    Z_list = []
    Recon_list = []

    with torch.no_grad():
        for i in range(0, len(X_tensor), BATCH_SIZE):
            xb = X_tensor[i:i + BATCH_SIZE].to(device)

            pred, z = model(xb)

            Z_list.append(z.cpu().numpy())
            Recon_list.append(pred.cpu().numpy())

    Z = np.vstack(Z_list)
    Recon_s = np.vstack(Recon_list)
    Recon = scaler.inverse_transform(Recon_s)

    return Z, Recon


# ============================================================
# 8) REGRESSOR LATENTE -> PARÂMETROS
# ============================================================

def treinar_regressor_parametros(
    df,
    fcols,
    fhz,
    y_ref_healthy,
    ae_extra
):
    df_params = construir_dataset_parametros_saudaveis(
        df=df,
        fcols=fcols,
        fhz=fhz,
        y_ref_healthy=y_ref_healthy
    )

    Z_all, Recon_all = codificar_curvas_ae(df, fcols, ae_extra)

    healthy_indices = df_params["index_original"].to_numpy(int)

    Z_h = Z_all[healthy_indices]

    P = df_params[["tau", "gain", "offset", "tilt"]].to_numpy(float)

    if USE_KNOWN_TEMPERATURE_FOR_PARAMS:
        # Modo controle: parâmetros previstos diretamente pela temperatura medida.
        # Útil para ver o limite físico do método.
        T = df_params["temperatura_c"].to_numpy(float).reshape(-1, 1)

        reg = make_pipeline(
            SkStandardScaler(),
            Ridge(alpha=RIDGE_ALPHA)
        )

        reg.fit(T, P)

        input_mode = "temperature"
    else:
        # Modo principal para pesquisa:
        # parâmetros previstos pelo espaço latente do autoencoder.
        reg = make_pipeline(
            SkStandardScaler(),
            Ridge(alpha=RIDGE_ALPHA)
        )

        reg.fit(Z_h, P)

        input_mode = "latent"

    return {
        "reg": reg,
        "df_params": df_params,
        "input_mode": input_mode
    }


def compensar_ae_parametrico(
    df,
    fcols,
    fhz,
    ae_extra,
    param_extra,
    y_ref_healthy
):
    print("\n====================================================")
    print("APLICANDO COMPENSAÇÃO AE + PARAMÉTRICA")
    print("====================================================")

    print("A curva compensada NÃO é gerada pelo decoder.")
    print("A referência NÃO é somada ponto a ponto.")
    print("Apenas tau, ganho, offset e inclinação são aplicados.")

    X = df[fcols].to_numpy(float)

    Z_all, Recon_all = codificar_curvas_ae(df, fcols, ae_extra)

    if param_extra["input_mode"] == "temperature":
        T_all = df["temperatura_c"].to_numpy(float).reshape(-1, 1)
        P_pred = param_extra["reg"].predict(T_all)
    else:
        P_pred = param_extra["reg"].predict(Z_all)

    ref_amp = np.ptp(y_ref_healthy) + 1e-12

    Y_comp = np.zeros_like(X)

    for i in range(len(X)):
        params = limitar_parametros(P_pred[i], ref_amp)

        y_full = aplicar_transformacao_parametrica(
            x=X[i],
            fHz=fhz,
            params=params
        )

        # Mistura conservadora:
        # evita que a transformação modifique demais a curva original.
        Y_comp[i] = X[i] + ALPHA_COMP * (y_full - X[i])

    df_comp = df.copy()
    df_comp[fcols] = Y_comp

    # Evita PerformanceWarning do pandas:
    # em vez de inserir uma coluna por vez, juntamos todas de uma vez.
    df_comp = pd.concat(
        [
            df_comp.copy(),
            pd.DataFrame(
                {
                    "tau_pred": P_pred[:, 0],
                    "gain_pred": P_pred[:, 1],
                    "offset_pred": P_pred[:, 2],
                    "tilt_pred": P_pred[:, 3],
                },
                index=df_comp.index
            )
        ],
        axis=1
    )

    df_recon = df.copy()
    df_recon[fcols] = Recon_all

    return df_comp, df_recon, Z_all, P_pred


# ============================================================
# 9) PARK
# ============================================================

def park_single(x, ref, fHz):
    df_band = fHz[-1] - fHz[0]
    tau_max = PARK_MAX_SHIFT_FRAC * df_band

    best_err = np.inf
    best_tau = 0.0
    best_dS = 0.0

    taus = np.linspace(-tau_max, tau_max, PARK_NSTEPS)

    for tau in taus:
        x_shift = shift_interp(x, fHz, tau)

        dS = np.mean(ref - x_shift)

        y_try = x_shift + dS

        err = np.mean((ref - y_try) ** 2)

        if err < best_err:
            best_err = err
            best_tau = tau
            best_dS = dS

    y_comp = shift_interp(x, fHz, best_tau) + best_dS
    y_comp = moving_average(y_comp, PARK_SMOOTH_WIN)

    return y_comp


def compensar_park(df, fcols, fHz, y_ref_healthy):
    print("\n====================================================")
    print("APLICANDO PARK")
    print("====================================================")

    X_all = df[fcols].to_numpy(float)

    Y_comp = np.zeros_like(X_all)

    for i in range(len(X_all)):
        Y_comp[i] = park_single(
            x=X_all[i],
            ref=y_ref_healthy,
            fHz=fHz
        )

        if (i + 1) % 20 == 0 or (i + 1) == len(X_all):
            print(f"Park: {i+1}/{len(X_all)} curvas compensadas")

    df_comp = df.copy()
    df_comp[fcols] = Y_comp

    return df_comp


# ============================================================
# 10) RESUMOS
# ============================================================

def resumo_geral(df_long):
    tabela = (
        df_long
        .groupby(["Metodo", "falha"])[[
            "RMSD",
            "CCDM",
            "DamageResidual_RMSD",
            "DamageResidual_CCDM",
            "Alteracao_RMSD"
        ]]
        .agg(["mean", "std", "min", "max"])
        .round(6)
    )

    return tabela


def resumo_por_temperatura(df_long):
    tabela = (
        df_long
        .groupby(["Metodo", "temperatura_c", "falha"])[[
            "RMSD",
            "CCDM",
            "DamageResidual_RMSD",
            "DamageResidual_CCDM",
            "Alteracao_RMSD"
        ]]
        .mean()
        .reset_index()
        .sort_values(["Metodo", "temperatura_c", "falha"])
    )

    return tabela


def checar_monotonicidade(df_long, metodos=("Original", "Park", "AE paramétrico")):
    df_use = df_long[df_long["Metodo"].isin(metodos)].copy()

    registros = []

    for metodo in sorted(df_use["Metodo"].unique()):
        df_m = df_use[df_use["Metodo"] == metodo]

        temps = sorted(df_m["temperatura_c"].unique())

        for T in temps:
            df_t = df_m[np.isclose(df_m["temperatura_c"], T)]

            danos_presentes = set(df_t["falha"].unique())

            if not {0, 1, 2}.issubset(danos_presentes):
                continue

            for metrica in ["RMSD", "CCDM"]:
                medias = {}

                for d in [0, 1, 2]:
                    medias[d] = df_t.loc[
                        df_t["falha"] == d,
                        metrica
                    ].mean()

                ok = medias[0] < medias[1] < medias[2]

                registros.append({
                    "Metodo": metodo,
                    "Temperatura": T,
                    "Metrica": metrica,
                    "D0": medias[0],
                    "D1": medias[1],
                    "D2": medias[2],
                    "Monotonico_D0_D1_D2": ok
                })

    df_mono = pd.DataFrame(registros)

    if len(df_mono) == 0:
        return df_mono, pd.DataFrame()

    resumo = (
        df_mono
        .groupby(["Metodo", "Metrica"])["Monotonico_D0_D1_D2"]
        .mean()
        .mul(100)
        .reset_index()
        .rename(columns={
            "Monotonico_D0_D1_D2": "Percentual_monotonico_%"
        })
    )

    return df_mono, resumo


# ============================================================
# 11) GRÁFICOS
# ============================================================

def selecionar_indice_por_dano_temperatura(
    df,
    falha,
    temperatura,
    ocorrencia=0
):
    df_d = df[df["falha"] == falha].copy()

    if len(df_d) == 0:
        raise ValueError(f"Nenhuma curva encontrada para falha = {falha}")

    temps_disponiveis = np.array(
        sorted(df_d["temperatura_c"].unique()),
        dtype=float
    )

    temp_usada = temps_disponiveis[
        np.argmin(np.abs(temps_disponiveis - temperatura))
    ]

    df_dt = df_d[np.isclose(df_d["temperatura_c"], temp_usada)].copy()

    if len(df_dt) == 0:
        raise ValueError(
            f"Nenhuma curva encontrada para falha={falha} "
            f"na temperatura {temp_usada} °C."
        )

    if ocorrencia >= len(df_dt):
        print(
            f"AVISO: falha={falha}, T={temp_usada} °C possui apenas "
            f"{len(df_dt)} curva(s). Usando ocorrência 0."
        )
        ocorrencia = 0

    idx = df_dt.index[ocorrencia]

    return idx, temp_usada


def plot_curvas_ae_parametrico(
    df_base,
    df_ae,
    df_recon,
    y_ref_healthy,
    healthy_by_temp,
    healthy_temps,
    fcols,
    fhz,
    temperatura_escolhida=55,
    danos=(0, 1, 2),
    ocorrencia=0,
    salvar=True,
    show=True
):
    aplicar_estilo_artigo()

    fhz_khz = fhz / 1e3

    fig, axes = plt.subplots(
        len(danos),
        1,
        figsize=(13, 5.7 * len(danos)),
        dpi=300,
        sharex=True
    )

    if len(danos) == 1:
        axes = [axes]

    print("\n====================================================")
    print("CURVAS — AE PARAMÉTRICO")
    print("====================================================")

    for ax, dano in zip(axes, danos):
        idx_show, temp_usada = selecionar_indice_por_dano_temperatura(
            df=df_base,
            falha=dano,
            temperatura=temperatura_escolhida,
            ocorrencia=ocorrencia
        )

        y_original = df_base.loc[idx_show, fcols].to_numpy(float)
        y_ae = df_ae.loc[idx_show, fcols].to_numpy(float)
        y_recon = df_recon.loc[idx_show, fcols].to_numpy(float)

        h_T, T_h_used = get_nearest_healthy_curve(
            healthy_by_temp=healthy_by_temp,
            healthy_temps=healthy_temps,
            temperatura=temp_usada
        )

        assinatura_original = y_original - h_T
        assinatura_ae = y_ae - y_ref_healthy

        print("\n----------------------------------------------------")
        print(f"Dano {dano}")
        print(f"Índice usado: {idx_show}")
        print(f"Temperatura da curva: {temp_usada} °C")
        print(f"Saudável real mais próxima: {T_h_used} °C")
        print("Original contra saudável REF:")
        print(f"  RMSD = {rmsd(y_original, y_ref_healthy):.6f}")
        print(f"  CCDM = {ccdm(y_original, y_ref_healthy):.6f}")
        print("AE paramétrico contra saudável REF:")
        print(f"  RMSD = {rmsd(y_ae, y_ref_healthy):.6f}")
        print(f"  CCDM = {ccdm(y_ae, y_ref_healthy):.6f}")
        print("Preservação da assinatura:")
        print(f"  RMSD residual = {rmsd(assinatura_ae, assinatura_original):.6f}")
        print(f"  CCDM residual = {ccdm(assinatura_ae, assinatura_original):.6f}")

        ax.plot(
            fhz_khz,
            y_ref_healthy,
            "--",
            color="black",
            linewidth=1.5,
            label=f"Referência saudável {REF_TEMP} °C"
        )

        ax.plot(
            fhz_khz,
            h_T,
            "-.",
            color="gray",
            linewidth=1.2,
            label=f"Saudável real {formatar_temp(T_h_used)} °C"
        )

        ax.plot(
            fhz_khz,
            y_recon,
            color="tab:green",
            linewidth=1.2,
            alpha=0.70,
            label="Reconstrução do AE, não usada na compensação"
        )

        ax.plot(
            fhz_khz,
            y_original,
            color="tab:red",
            linewidth=1.2,
            alpha=0.75,
            label=f"Original — Dano {dano} — {formatar_temp(temp_usada)} °C"
        )

        ax.plot(
            fhz_khz,
            y_ae,
            color="tab:blue",
            linewidth=2.0,
            label="AE paramétrico compensado"
        )

        ax.set_ylabel("Parte real da impedância")

        ax.set_title(
            f"Dano {dano} — {formatar_temp(temp_usada)} °C "
            f"→ compensação paramétrica para {REF_TEMP} °C"
        )

        ax.grid(False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        ax.legend(frameon=True, facecolor="white", edgecolor="none")

    axes[-1].set_xlabel("Frequência (kHz)")

    plt.tight_layout()

    if salvar:
        nome = (
            f"Curvas_AE_parametrico_"
            f"Temp_{formatar_temp(temperatura_escolhida)}C"
        )

        salvar_figura(fig, nome)

    if show:
        plt.show()
    else:
        plt.close(fig)


def plot_ae_loss(history, salvar=True, show=True):
    aplicar_estilo_artigo()

    fig, ax = plt.subplots(figsize=(10, 5), dpi=300)

    ax.plot(history["epoch"], history["train_loss"], linewidth=2, label="Treino")
    ax.plot(history["epoch"], history["val_loss"], linewidth=2, label="Validação")
    ax.plot(history["epoch"], history["val_recon"], linewidth=2, label="Reconstrução")

    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Treinamento do Autoencoder saudável")

    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.legend(frameon=True, facecolor="white", edgecolor="none")

    plt.tight_layout()

    if salvar:
        salvar_figura(fig, "AE_loss_parametrico")

    if show:
        plt.show()
    else:
        plt.close(fig)


def escolher_temperaturas_validas(
    df_long,
    metodos=("Original", "Park", "AE paramétrico"),
    n_temps=8,
    seed=42,
    temperaturas_especificas=None
):
    valid_temps = None

    for metodo in metodos:
        for dano in [0, 1, 2]:
            temps = set(
                df_long.loc[
                    (df_long["Metodo"] == metodo) &
                    (df_long["falha"] == dano),
                    "temperatura_c"
                ].unique()
            )

            if valid_temps is None:
                valid_temps = temps
            else:
                valid_temps = valid_temps & temps

    valid_temps = sorted(list(valid_temps))

    if len(valid_temps) == 0:
        raise ValueError("Nenhuma temperatura contém os três danos.")

    if temperaturas_especificas is not None:
        out = []

        for T in temperaturas_especificas:
            if any(np.isclose(T, Tv) for Tv in valid_temps):
                arr = np.asarray(valid_temps, dtype=float)
                out.append(float(arr[np.argmin(np.abs(arr - T))]))

        valid_temps = sorted(list(set(out)))

        if len(valid_temps) == 0:
            raise ValueError("Nenhuma temperatura escolhida é válida.")

        return valid_temps

    if len(valid_temps) > n_temps:
        rng = np.random.default_rng(seed)
        valid_temps = sorted(rng.choice(valid_temps, n_temps, replace=False))

    return valid_temps


def valores_medios_por_temp_dano_metodo(df_long, metodo, dano, temperaturas, metrica):
    vals = []

    for T in temperaturas:
        mask = (
            (df_long["Metodo"] == metodo) &
            (df_long["falha"] == dano) &
            (np.isclose(df_long["temperatura_c"], T))
        )

        if np.any(mask):
            vals.append(df_long.loc[mask, metrica].mean())
        else:
            vals.append(np.nan)

    return vals


def plot_barras_metodos(
    df_long,
    metricas=("RMSD", "CCDM"),
    temperaturas=None,
    n_temps=8,
    seed=42,
    nome_base="Barras_Original_Park_AE_Parametrico",
    salvar=True,
    show=True
):
    aplicar_estilo_artigo()

    if USAR_PARK:
        metodos = ["Original", "Park", "AE paramétrico"]
    else:
        metodos = ["Original", "AE paramétrico"]

    danos = [0, 1, 2]

    df_use = df_long[df_long["Metodo"].isin(metodos)].copy()

    temps_validas = escolher_temperaturas_validas(
        df_long=df_use,
        metodos=metodos,
        n_temps=n_temps,
        seed=seed,
        temperaturas_especificas=temperaturas
    )

    x = np.arange(len(temps_validas))

    bar_w = 0.08
    gap = 0.08

    offsets_por_metodo = {}
    start = 0.0

    for metodo in metodos:
        offsets_por_metodo[metodo] = start + np.array([0, 1, 2]) * bar_w
        start += 3 * bar_w + gap

    all_offsets = np.concatenate(list(offsets_por_metodo.values()))
    x_center = x + np.mean(all_offsets)

    colors = {
        0: "tab:blue",
        1: "tab:orange",
        2: "tab:red"
    }

    alphas = {
        "Original": 0.35,
        "Park": 0.65,
        "AE paramétrico": 1.00
    }

    fig, axes = plt.subplots(
        1,
        len(metricas),
        figsize=(11 * len(metricas), 7.2),
        dpi=300
    )

    if len(metricas) == 1:
        axes = [axes]

    for ax in axes:
        ax.grid(False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    for j, metrica in enumerate(metricas):
        ax = axes[j]

        for metodo in metodos:
            for i, dano in enumerate(danos):
                vals = valores_medios_por_temp_dano_metodo(
                    df_long=df_use,
                    metodo=metodo,
                    dano=dano,
                    temperaturas=temps_validas,
                    metrica=metrica
                )

                ax.bar(
                    x + offsets_por_metodo[metodo][i],
                    vals,
                    width=bar_w,
                    color=colors[dano],
                    alpha=alphas.get(metodo, 1.0),
                    edgecolor="black",
                    linewidth=0.7,
                    label=f"{metodo} — Dano {dano}" if j == 0 else None
                )

        ax.set_ylabel(metrica)
        ax.set_xlabel("Temperatura (°C)", labelpad=10)

        ax.set_xticks(x_center)
        ax.set_xticklabels([formatar_temp(T) for T in temps_validas])

        letra = chr(ord("a") + j)

        ax.text(
            0.5,
            -0.30,
            f"({letra}) {metrica}",
            transform=ax.transAxes,
            ha="center",
            va="top",
            fontsize=22
        )

    handles, labels = axes[0].get_legend_handles_labels()

    fig.legend(
        handles,
        labels,
        loc="upper center",
        ncol=3,
        frameon=False,
        bbox_to_anchor=(0.5, 1.12)
    )

    plt.tight_layout(rect=[0, 0.10, 1, 0.88])

    if salvar:
        salvar_figura(fig, nome_base)

    if show:
        plt.show()
    else:
        plt.close(fig)


# ============================================================
# 12) EXECUÇÃO PRINCIPAL
# ============================================================

def executar_ae_parametrico():
    timings = {}

    print("====================================================")
    print("AUTOENCODER PARAMÉTRICO PARA COMPENSAÇÃO TÉRMICA")
    print("====================================================")

    t0 = time.time()

    df = pd.read_pickle(ARQ_BASE).reset_index(drop=True)

    required_cols = {"temperatura_c", "falha"}

    missing = required_cols - set(df.columns)

    if len(missing) > 0:
        raise ValueError(f"Colunas obrigatórias ausentes: {missing}")

    fcols, fhz = get_freq_columns(df, FREQ_MIN_KHZ, FREQ_MAX_KHZ)

    if len(fcols) == 0:
        raise ValueError("Nenhuma coluna de frequência encontrada na faixa escolhida.")

    healthy_by_temp, healthy_temps, y_ref_healthy, ref_temp_used = (
        get_healthy_references_by_temperature(
            df=df,
            fcols=fcols,
            ref_temp=REF_TEMP
        )
    )

    timings["load_reference"] = time.time() - t0

    print(f"\nTotal de amostras: {len(df)}")
    print(f"Classes de dano: {sorted(df['falha'].unique())}")
    print(f"Amostras saudáveis: {len(df[df['falha'] == 0])}")
    print(f"Faixa usada: {FREQ_MIN_KHZ}-{FREQ_MAX_KHZ} kHz")
    print(f"Número de pontos de frequência: {len(fcols)}")
    print(f"Temperatura de referência desejada: {REF_TEMP} °C")
    print(f"Temperatura saudável usada como REF: {ref_temp_used} °C")

    print("\n==================== CONFIGURAÇÃO CONSERVADORA ====================")
    print(f"EPOCHS = {EPOCHS}")
    print(f"LATENT_DIM = {LATENT_DIM}")
    print(f"SHIFT_MAX_FRAC = {SHIFT_MAX_FRAC}")
    print(f"GAIN_MIN / GAIN_MAX = {GAIN_MIN} / {GAIN_MAX}")
    print(f"OFFSET_FRAC_LIMIT = {OFFSET_FRAC_LIMIT}")
    print(f"TILT_FRAC_LIMIT = {TILT_FRAC_LIMIT}")
    print(f"ALPHA_COMP = {ALPHA_COMP}")
    print(f"RIDGE_ALPHA = {RIDGE_ALPHA}")

    # ---------------- ORIGINAL ----------------

    t0 = time.time()

    df_original_metricas = calcular_metricas_com_preservacao(
        df_curvas=df,
        df_original=df,
        fcols=fcols,
        y_ref_healthy=y_ref_healthy,
        healthy_by_temp=healthy_by_temp,
        healthy_temps=healthy_temps,
        metodo="Original"
    )

    timings["original_metrics"] = time.time() - t0

    # ---------------- PARK ----------------

    if USAR_PARK:
        t0 = time.time()

        df_park_curvas = compensar_park(
            df=df,
            fcols=fcols,
            fHz=fhz,
            y_ref_healthy=y_ref_healthy
        )

        df_park_metricas = calcular_metricas_com_preservacao(
            df_curvas=df_park_curvas,
            df_original=df,
            fcols=fcols,
            y_ref_healthy=y_ref_healthy,
            healthy_by_temp=healthy_by_temp,
            healthy_temps=healthy_temps,
            metodo="Park"
        )

        timings["park"] = time.time() - t0
    else:
        df_park_curvas = None
        df_park_metricas = None

    # ---------------- AE PARAMÉTRICO ----------------

    t0 = time.time()

    ae_extra = treinar_autoencoder_saudavel(df, fcols)

    param_extra = treinar_regressor_parametros(
        df=df,
        fcols=fcols,
        fhz=fhz,
        y_ref_healthy=y_ref_healthy,
        ae_extra=ae_extra
    )

    df_ae_curvas, df_recon_curvas, ae_latent, params_pred = compensar_ae_parametrico(
        df=df,
        fcols=fcols,
        fhz=fhz,
        ae_extra=ae_extra,
        param_extra=param_extra,
        y_ref_healthy=y_ref_healthy
    )

    df_ae_metricas = calcular_metricas_com_preservacao(
        df_curvas=df_ae_curvas,
        df_original=df,
        fcols=fcols,
        y_ref_healthy=y_ref_healthy,
        healthy_by_temp=healthy_by_temp,
        healthy_temps=healthy_temps,
        metodo="AE paramétrico"
    )

    timings["ae_parametrico"] = time.time() - t0

    # ---------------- JUNTAR ----------------

    partes = [df_original_metricas]

    if USAR_PARK and df_park_metricas is not None:
        partes.append(df_park_metricas)

    partes.append(df_ae_metricas)

    df_long = pd.concat(partes, axis=0, ignore_index=True)

    tabela_resumo = resumo_geral(df_long)
    tabela_temp = resumo_por_temperatura(df_long)

    metodos_mono = ["Original", "AE paramétrico"]

    if USAR_PARK:
        metodos_mono.insert(1, "Park")

    df_mono, resumo_mono = checar_monotonicidade(
        df_long,
        metodos=tuple(metodos_mono)
    )

    # ---------------- SALVAR ----------------

    df_long.to_csv(os.path.join(OUTPUT_DIR, "df_long_metricas.csv"), index=False)
    tabela_temp.to_csv(os.path.join(OUTPUT_DIR, "resumo_por_temperatura.csv"), index=False)
    df_mono.to_csv(os.path.join(OUTPUT_DIR, "monotonicidade.csv"), index=False)
    resumo_mono.to_csv(os.path.join(OUTPUT_DIR, "resumo_monotonicidade.csv"), index=False)

    ae_extra["history"].to_csv(
        os.path.join(OUTPUT_DIR, "historico_loss_ae.csv"),
        index=False
    )

    param_extra["df_params"].to_csv(
        os.path.join(OUTPUT_DIR, "parametros_ajustados_saudaveis.csv"),
        index=False
    )

    pd.DataFrame(ae_latent).to_csv(
        os.path.join(OUTPUT_DIR, "latent_autoencoder.csv"),
        index=False
    )

    pd.DataFrame(
        params_pred,
        columns=["tau_pred", "gain_pred", "offset_pred", "tilt_pred"]
    ).to_csv(
        os.path.join(OUTPUT_DIR, "parametros_preditos_todas_curvas.csv"),
        index=False
    )

    print("\n==================== RESUMO GERAL ====================")
    print(tabela_resumo)

    print("\n==================== MONOTONICIDADE ====================")
    print(resumo_mono)

    print("\n==================== TEMPOS ====================")
    for k, v in timings.items():
        print(f"{k:28s}: {v:.3f} s")

    print("\n✅ Execução concluída.")

    return {
        "df_base": df,
        "df_park_curvas": df_park_curvas,
        "df_ae_curvas": df_ae_curvas,
        "df_recon_curvas": df_recon_curvas,
        "df_original_metricas": df_original_metricas,
        "df_park_metricas": df_park_metricas,
        "df_ae_metricas": df_ae_metricas,
        "df_long": df_long,
        "tabela_resumo": tabela_resumo,
        "tabela_temp": tabela_temp,
        "df_mono": df_mono,
        "resumo_mono": resumo_mono,
        "healthy_by_temp": healthy_by_temp,
        "healthy_temps": healthy_temps,
        "y_ref_healthy": y_ref_healthy,
        "ref_temp_used": ref_temp_used,
        "fcols": fcols,
        "fhz": fhz,
        "ae_extra": ae_extra,
        "param_extra": param_extra,
        "ae_history": ae_extra["history"],
        "ae_latent": ae_latent,
        "params_pred": params_pred,
        "timings": timings
    }


# ============================================================
# 13) RODAR TUDO
# ============================================================

resultados = executar_ae_parametrico()

df_base = resultados["df_base"]
df_park_curvas = resultados["df_park_curvas"]
df_ae_curvas = resultados["df_ae_curvas"]
df_recon_curvas = resultados["df_recon_curvas"]

df_original_metricas = resultados["df_original_metricas"]
df_park_metricas = resultados["df_park_metricas"]
df_ae_metricas = resultados["df_ae_metricas"]

df_long = resultados["df_long"]

tabela_resumo = resultados["tabela_resumo"]
tabela_temp = resultados["tabela_temp"]
df_mono = resultados["df_mono"]
resumo_mono = resultados["resumo_mono"]

healthy_by_temp = resultados["healthy_by_temp"]
healthy_temps = resultados["healthy_temps"]
y_ref_healthy = resultados["y_ref_healthy"]
ref_temp_used = resultados["ref_temp_used"]

fcols = resultados["fcols"]
fhz = resultados["fhz"]

ae_extra = resultados["ae_extra"]
param_extra = resultados["param_extra"]
ae_history = resultados["ae_history"]
ae_latent = resultados["ae_latent"]
params_pred = resultados["params_pred"]
timings = resultados["timings"]


# ============================================================
# 14) GERAR FIGURAS
# ============================================================

plot_curvas_ae_parametrico(
    df_base=df_base,
    df_ae=df_ae_curvas,
    df_recon=df_recon_curvas,
    y_ref_healthy=y_ref_healthy,
    healthy_by_temp=healthy_by_temp,
    healthy_temps=healthy_temps,
    fcols=fcols,
    fhz=fhz,
    temperatura_escolhida=TEMP_ESCOLHIDA,
    danos=DANOS_PLOTAR,
    ocorrencia=OCORRENCIA_CURVA,
    salvar=True,
    show=True
)

plot_barras_metodos(
    df_long=df_long,
    metricas=("RMSD", "CCDM"),
    temperaturas=TEMPERATURAS_BARRAS,
    n_temps=N_TEMPS_BARRAS,
    seed=42,
    nome_base="Barras_Original_Park_AE_Parametrico_RMSD_CCDM",
    salvar=True,
    show=True
)

plot_barras_metodos(
    df_long=df_long,
    metricas=("DamageResidual_RMSD", "DamageResidual_CCDM"),
    temperaturas=TEMPERATURAS_BARRAS,
    n_temps=N_TEMPS_BARRAS,
    seed=42,
    nome_base="Barras_Preservacao_Assinatura_AE_Parametrico",
    salvar=True,
    show=True
)

plot_ae_loss(
    history=ae_history,
    salvar=True,
    show=True
)


# ============================================================
# 15) MOSTRAR TABELAS
# ============================================================

print("\n==================== TABELA RESUMO ====================")
display(tabela_resumo)

print("\n==================== RESUMO POR TEMPERATURA ====================")
display(tabela_temp)

print("\n==================== MONOTONICIDADE POR TEMPERATURA ====================")
display(df_mono)

print("\n==================== RESUMO DA MONOTONICIDADE ====================")
display(resumo_mono)

print("\n✅ Todos os gráficos foram gerados e salvos.")
print(f"Pasta de saída: {OUTPUT_DIR}")

print("\nObservação importante:")
print("A linha verde dos gráficos é a reconstrução do AE, mas ela NÃO é usada como curva compensada.")
print("Ela aparece só para diagnóstico. A curva compensada é gerada pela transformação paramétrica.")
